SEC Financial Dashboard Project

This project builds a financial performance dataset using SEC Financial Statement Data Sets from 2020 to 2025. The objective is to extract, clean, transform, and prepare real public company financial data for a Power BI dashboard.

The analysis focuses on five major U.S. companies:
Apple
Microsoft
Amazon
Tesla
JPMorgan Chase

In [8]:
import pandas as pd
import numpy as np
import os

In [10]:
base_path = os.path.expanduser("/Users/ankitatripathy/Desktop/PowerBi/Raw Data")

print("Folder exists:", os.path.exists(base_path))
print(sorted(os.listdir(base_path)))

Folder exists: True
['.DS_Store', '2020q1', '2020q2', '2020q3', '2020q4', '2021q1', '2021q2', '2021q3', '2021q4', '2022q1', '2022q2', '2022q3', '2022q4', '2023q1', '2023q2', '2023q3', '2024q1', '2024q2', '2024q3', '2024q4', '2025q1', '2025q2', '2025q3', '2025q4']


In [12]:
target_ciks = [
    320193,     # Apple
    789019,    # Microsoft
    1018724,   # Amazon
    1318605,   # Tesla
    19617      # JPMorgan Chase
]

financial_tags = [
    "Revenues",
    "SalesRevenueNet",
    "RevenueFromContractWithCustomerExcludingAssessedTax",

    # Banking revenue tags for JPMorgan Chase
    "RevenuesNetOfInterestExpense",
    "InterestIncomeExpenseNet",
    "NoninterestIncome",

    "GrossProfit",
    "OperatingIncomeLoss",
    "NetIncomeLoss",
    "Assets",
    "Liabilities",
    "StockholdersEquity",
    "CashAndCashEquivalentsAtCarryingValue",
    "NetCashProvidedByUsedInOperatingActivities"
]

In [14]:
master_list = []

quarter_folders = sorted([
    folder for folder in os.listdir(base_path)
    if os.path.isdir(os.path.join(base_path, folder))
])

for folder in quarter_folders:
    print("Processing:", folder)

    sub_path = os.path.join(base_path, folder, "sub.txt")
    num_path = os.path.join(base_path, folder, "num.txt")

    if not os.path.exists(sub_path) or not os.path.exists(num_path):
        print("Missing files:", folder)
        continue

    sub = pd.read_csv(sub_path, sep="\t", low_memory=False)

    sub["cik"] = pd.to_numeric(sub["cik"], errors="coerce")

    sub_filtered = sub[
        (sub["cik"].isin(target_ciks)) &
        (sub["form"].isin(["10-K", "10-Q"]))
    ].copy()

    if sub_filtered.empty:
        continue

    needed_adsh = sub_filtered["adsh"].unique()

    num = pd.read_csv(num_path, sep="\t", low_memory=False)

    num_filtered = num[
        (num["adsh"].isin(needed_adsh)) &
        (num["tag"].isin(financial_tags)) &
        (num["uom"] == "USD") &
        (num["segments"].isna())
    ].copy()

    if num_filtered.empty:
        continue

    merged = pd.merge(
        num_filtered,
        sub_filtered[
            [
                "adsh",
                "name",
                "cik",
                "period",
                "filed",
                "form"
            ]
        ],
        on="adsh",
        how="inner"
    )

    merged["QuarterFolder"] = folder

    master_list.append(merged)

master = pd.concat(master_list, ignore_index=True)

print("Master shape:", master.shape)
master.head()

Processing: 2020q1
Processing: 2020q2
Processing: 2020q3
Processing: 2020q4
Processing: 2021q1
Processing: 2021q2
Processing: 2021q3
Processing: 2021q4
Processing: 2022q1
Processing: 2022q2
Processing: 2022q3
Processing: 2022q4
Processing: 2023q1
Processing: 2023q2
Processing: 2023q3
Processing: 2024q1
Processing: 2024q2
Processing: 2024q3
Processing: 2024q4
Processing: 2025q1
Processing: 2025q2
Processing: 2025q3
Processing: 2025q4
Master shape: (3028, 16)


,adsh,tag,version,ddate,qtrs,uom,segments,coreg,value,footnote,name,cik,period,filed,form,QuarterFolder
0,0001018724-20-000004,RevenueFromContractWithCustomerExcludingAssess...,us-gaap/2019,20180930,1,USD,NaN,NaN,5.657600e+10,NaN,AMAZON COM INC,1018724,20191231.0,20200131,10-K,2020q1
1,0000019617-20-000257,NoninterestIncome,us-gaap/2019,20181231,4,USD,NaN,NaN,5.397000e+10,NaN,JPMORGAN CHASE & CO,19617,20191231.0,20200225,10-K,2020q1
2,0001564590-20-002450,CashAndCashEquivalentsAtCarryingValue,us-gaap/2019,20190630,0,USD,NaN,NaN,1.135600e+10,NaN,MICROSOFT CORP,789019,20191231.0,20200129,10-Q,2020q1
3,0001564590-20-004475,NetIncomeLoss,us-gaap/2019,20171231,4,USD,NaN,NaN,-1.962000e+09,NaN,"TESLA, INC.",1318605,20191231.0,20200213,10-K,2020q1
4,0001564590-20-004475,CashAndCashEquivalentsAtCarryingValue,us-gaap/2019,20191231,0,USD,NaN,NaN,6.268000e+09,NaN,"TESLA, INC.",1318605,20191231.0,20200213,10-K,2020q1


In [16]:
master["ddate"] = pd.to_datetime(
    master["ddate"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

master["filed"] = pd.to_datetime(
    master["filed"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

master["Year"] = master["ddate"].dt.year
master["Quarter"] = master["ddate"].dt.quarter

master = master[
    (master["Year"] >= 2020) &
    (master["Year"] <= 2026)
].copy()

print("Master after date filter:", master.shape)
master.head()

Master after date filter: (2486, 18)


,adsh,tag,version,ddate,qtrs,uom,segments,coreg,value,footnote,name,cik,period,filed,form,QuarterFolder,Year,Quarter
169,0000019617-20-000299,InterestIncomeExpenseNet,us-gaap/2019,2020-03-31,1,USD,NaN,NaN,1.443900e+10,NaN,JPMORGAN CHASE & CO,19617,20200331.0,2020-05-07,10-Q,2020q2,2020,1
171,0000019617-20-000299,NetCashProvidedByUsedInOperatingActivities,us-gaap/2019,2020-03-31,1,USD,NaN,NaN,-1.207720e+11,NaN,JPMORGAN CHASE & CO,19617,20200331.0,2020-05-07,10-Q,2020q2,2020,1
172,0001018724-20-000010,StockholdersEquity,us-gaap/2019,2020-03-31,0,USD,NaN,NaN,6.527200e+10,NaN,AMAZON COM INC,1018724,20200331.0,2020-05-01,10-Q,2020q2,2020,1
177,0001564590-20-019931,Liabilities,us-gaap/2019,2020-03-31,0,USD,NaN,NaN,2.651800e+10,NaN,"TESLA, INC.",1318605,20200331.0,2020-04-30,10-Q,2020q2,2020,1
179,0000019617-20-000299,Liabilities,us-gaap/2019,2020-03-31,0,USD,NaN,NaN,2.878169e+12,The following table presents information on as...,JPMORGAN CHASE & CO,19617,20200331.0,2020-05-07,10-Q,2020q2,2020,1


In [18]:
financial_table = master.pivot_table(
    index=[
        "name",
        "cik",
        "form",
        "filed",
        "ddate",
        "Year",
        "Quarter"
    ],
    columns="tag",
    values="value",
    aggfunc="first"
).reset_index()

financial_table.columns.name = None

print("Financial table shape:", financial_table.shape)
financial_table.head()

Financial table shape: (378, 20)


,name,cik,form,filed,ddate,Year,Quarter,Assets,CashAndCashEquivalentsAtCarryingValue,GrossProfit,InterestIncomeExpenseNet,Liabilities,NetCashProvidedByUsedInOperatingActivities,NetIncomeLoss,NoninterestIncome,OperatingIncomeLoss,RevenueFromContractWithCustomerExcludingAssessedTax,Revenues,RevenuesNetOfInterestExpense,StockholdersEquity
0,AMAZON COM INC,1018724,10-K,2021-02-03,2020-03-31,2020,1,NaN,NaN,NaN,NaN,NaN,NaN,2.535000e+09,NaN,3.989000e+09,7.545200e+10,NaN,NaN,NaN
1,AMAZON COM INC,1018724,10-K,2021-02-03,2020-06-30,2020,2,NaN,NaN,NaN,NaN,NaN,NaN,5.243000e+09,NaN,5.843000e+09,8.891200e+10,NaN,NaN,NaN
2,AMAZON COM INC,1018724,10-K,2021-02-03,2020-09-30,2020,3,NaN,NaN,NaN,NaN,NaN,NaN,6.331000e+09,NaN,6.194000e+09,9.614500e+10,NaN,NaN,NaN
3,AMAZON COM INC,1018724,10-K,2021-02-03,2020-12-31,2020,4,3.211950e+11,4.212200e+10,NaN,NaN,NaN,6.606400e+10,7.222000e+09,NaN,2.289900e+10,1.255550e+11,NaN,NaN,9.340400e+10
4,AMAZON COM INC,1018724,10-K,2022-02-04,2020-12-31,2020,4,3.211950e+11,4.212200e+10,NaN,NaN,NaN,6.606400e+10,2.133100e+10,NaN,2.289900e+10,3.860640e+11,NaN,NaN,9.340400e+10


In [20]:
print(financial_table.columns.tolist())

['name', 'cik', 'form', 'filed', 'ddate', 'Year', 'Quarter', 'Assets', 'CashAndCashEquivalentsAtCarryingValue', 'GrossProfit', 'InterestIncomeExpenseNet', 'Liabilities', 'NetCashProvidedByUsedInOperatingActivities', 'NetIncomeLoss', 'NoninterestIncome', 'OperatingIncomeLoss', 'RevenueFromContractWithCustomerExcludingAssessedTax', 'Revenues', 'RevenuesNetOfInterestExpense', 'StockholdersEquity']


In [22]:
final_df = financial_table.copy()

final_df["Company"] = final_df["name"]

final_df["Revenue"] = np.nan

if "Revenues" in final_df.columns:
    final_df["Revenue"] = final_df["Revenue"].fillna(final_df["Revenues"])

if "SalesRevenueNet" in final_df.columns:
    final_df["Revenue"] = final_df["Revenue"].fillna(final_df["SalesRevenueNet"])

if "RevenueFromContractWithCustomerExcludingAssessedTax" in final_df.columns:
    final_df["Revenue"] = final_df["Revenue"].fillna(
        final_df["RevenueFromContractWithCustomerExcludingAssessedTax"]
    )

if "RevenuesNetOfInterestExpense" in final_df.columns:
    final_df["Revenue"] = final_df["Revenue"].fillna(
        final_df["RevenuesNetOfInterestExpense"]
    )

if "InterestIncomeExpenseNet" in final_df.columns:
    final_df["Revenue"] = final_df["Revenue"].fillna(
        final_df["InterestIncomeExpenseNet"]
    )

if "NoninterestIncome" in final_df.columns:
    final_df["Revenue"] = final_df["Revenue"].fillna(
        final_df["NoninterestIncome"]
    )

final_df["Gross_Profit"] = final_df.get("GrossProfit")
final_df["Operating_Income"] = final_df.get("OperatingIncomeLoss")
final_df["Net_Income"] = final_df.get("NetIncomeLoss")
final_df["Assets"] = final_df.get("Assets")
final_df["Liabilities"] = final_df.get("Liabilities")
final_df["Equity"] = final_df.get("StockholdersEquity")
final_df["Cash"] = final_df.get("CashAndCashEquivalentsAtCarryingValue")
final_df["Operating_Cash_Flow"] = final_df.get("NetCashProvidedByUsedInOperatingActivities")

In [24]:
company_map = {
    "APPLE INC": "Apple",
    "MICROSOFT CORP": "Microsoft",
    "AMAZON COM INC": "Amazon",
    "TESLA, INC.": "Tesla",
    "JPMORGAN CHASE & CO": "JPMorgan Chase"
}

final_df["Company"] = final_df["Company"].replace(company_map)

In [26]:
final_df["Profit_Margin"] = np.where(
    final_df["Revenue"] != 0,
    (final_df["Net_Income"] / final_df["Revenue"]) * 100,
    np.nan
)

final_df["Gross_Margin"] = np.where(
    final_df["Revenue"] != 0,
    (final_df["Gross_Profit"] / final_df["Revenue"]) * 100,
    np.nan
)

final_df["Operating_Margin"] = np.where(
    final_df["Revenue"] != 0,
    (final_df["Operating_Income"] / final_df["Revenue"]) * 100,
    np.nan
)

final_df["ROE"] = np.where(
    final_df["Equity"] != 0,
    (final_df["Net_Income"] / final_df["Equity"]) * 100,
    np.nan
)

final_df["Cash_Flow_Margin"] = np.where(
    final_df["Revenue"] != 0,
    (final_df["Operating_Cash_Flow"] / final_df["Revenue"]) * 100,
    np.nan
)

In [28]:
financial_performance_df = final_df[
    [
        "Company",
        "cik",
        "form",
        "filed",
        "ddate",
        "Year",
        "Quarter",
        "Revenue",
        "Gross_Profit",
        "Operating_Income",
        "Net_Income",
        "Operating_Cash_Flow",
        "Equity",
        "Profit_Margin",
        "Gross_Margin",
        "Operating_Margin",
        "ROE",
        "Cash_Flow_Margin"
    ]
].copy()

print("Full financial performance dataset:", financial_performance_df.shape)
financial_performance_df.head()

Full financial performance dataset: (378, 18)


,Company,cik,form,filed,ddate,Year,Quarter,Revenue,Gross_Profit,Operating_Income,Net_Income,Operating_Cash_Flow,Equity,Profit_Margin,Gross_Margin,Operating_Margin,ROE,Cash_Flow_Margin
0,Amazon,1018724,10-K,2021-02-03,2020-03-31,2020,1,7.545200e+10,NaN,3.989000e+09,2.535000e+09,NaN,NaN,3.359752,NaN,5.286805,NaN,NaN
1,Amazon,1018724,10-K,2021-02-03,2020-06-30,2020,2,8.891200e+10,NaN,5.843000e+09,5.243000e+09,NaN,NaN,5.896842,NaN,6.571666,NaN,NaN
2,Amazon,1018724,10-K,2021-02-03,2020-09-30,2020,3,9.614500e+10,NaN,6.194000e+09,6.331000e+09,NaN,NaN,6.584846,NaN,6.442353,NaN,NaN
3,Amazon,1018724,10-K,2021-02-03,2020-12-31,2020,4,1.255550e+11,NaN,2.289900e+10,7.222000e+09,6.606400e+10,9.340400e+10,5.752061,NaN,18.238222,7.732003,52.617578
4,Amazon,1018724,10-K,2022-02-04,2020-12-31,2020,4,3.860640e+11,NaN,2.289900e+10,2.133100e+10,6.606400e+10,9.340400e+10,5.525250,NaN,5.931400,22.837352,17.112189


In [30]:
quarterly_df = financial_performance_df[
    financial_performance_df["form"] == "10-Q"
].copy()

quarterly_df = quarterly_df.sort_values(
    ["Company", "Year", "Quarter", "filed"]
)

quarterly_df = quarterly_df.drop_duplicates(
    subset=["Company", "Year", "Quarter"],
    keep="last"
)

print("Quarterly dataset shape:", quarterly_df.shape)
quarterly_df.head()

Quarterly dataset shape: (114, 18)


,Company,cik,form,filed,ddate,Year,Quarter,Revenue,Gross_Profit,Operating_Income,Net_Income,Operating_Cash_Flow,Equity,Profit_Margin,Gross_Margin,Operating_Margin,ROE,Cash_Flow_Margin
25,Amazon,1018724,10-Q,2021-07-30,2020-03-31,2020,1,NaN,NaN,NaN,NaN,NaN,6.527200e+10,NaN,NaN,NaN,NaN,NaN
30,Amazon,1018724,10-Q,2021-10-29,2020-06-30,2020,2,NaN,NaN,NaN,NaN,NaN,7.372800e+10,NaN,NaN,NaN,NaN,NaN
31,Amazon,1018724,10-Q,2021-10-29,2020-09-30,2020,3,2.605090e+11,NaN,6.194000e+09,6.331000e+09,5.529200e+10,8.277500e+10,2.430242,NaN,2.377653,7.648445,21.224603
45,Amazon,1018724,10-Q,2022-10-28,2020-12-31,2020,4,NaN,NaN,NaN,NaN,NaN,9.340400e+10,NaN,NaN,NaN,NaN,NaN
40,Amazon,1018724,10-Q,2022-07-29,2021-03-31,2021,1,NaN,NaN,NaN,NaN,NaN,1.033200e+11,NaN,NaN,NaN,NaN,NaN


In [32]:
quarterly_df["Company"].value_counts()

Company
Amazon            23
JPMorgan Chase    23
Microsoft         23
Tesla             23
Apple             22
Name: count, dtype: int64

In [34]:
quarterly_df.notnull().sum().sort_values()

Gross_Profit            41
Gross_Margin            41
Operating_Margin        48
Operating_Income        48
ROE                     52
Profit_Margin           66
Operating_Cash_Flow     66
Net_Income              66
Cash_Flow_Margin        66
Revenue                 66
Equity                  96
Quarter                114
Year                   114
ddate                  114
filed                  114
form                   114
cik                    114
Company                114
dtype: int64

In [36]:
quarterly_df[
    quarterly_df["Company"] == "JPMorgan Chase"
][
    [
        "Company",
        "Year",
        "Quarter",
        "Revenue",
        "Net_Income",
        "Operating_Cash_Flow",
        "Equity"
    ]
]

,Company,Year,Quarter,Revenue,Net_Income,Operating_Cash_Flow,Equity
202,JPMorgan Chase,2020,1,2.828600e+10,2.865000e+09,-1.200890e+11,2.612620e+11
205,JPMorgan Chase,2020,2,3.307500e+10,7.552000e+09,-3.717000e+10,2.644660e+11
208,JPMorgan Chase,2020,3,2.925500e+10,1.699500e+10,-5.185800e+10,2.711130e+11
209,JPMorgan Chase,2020,4,NaN,NaN,NaN,2.793540e+11
211,JPMorgan Chase,2021,1,3.226600e+10,1.430000e+10,-4.387200e+10,2.807140e+11
214,JPMorgan Chase,2021,2,3.047900e+10,2.624800e+10,-3.034200e+10,2.863860e+11
217,JPMorgan Chase,2021,3,2.964700e+10,3.793500e+10,-7.011000e+09,2.900410e+11
218,JPMorgan Chase,2021,4,NaN,NaN,NaN,2.941270e+11
220,JPMorgan Chase,2022,1,3.071700e+10,8.282000e+09,-4.191700e+10,2.858990e+11
223,JPMorgan Chase,2022,2,3.071500e+10,1.693100e+10,2.410100e+10,2.861430e+11


In [38]:
powerbi_df = quarterly_df.copy()

powerbi_df["Sector"] = powerbi_df["Company"].map({
    "Amazon": "Technology",
    "Apple": "Technology",
    "Microsoft": "Technology",
    "Tesla": "Technology",
    "JPMorgan Chase": "Banking"
})

print(powerbi_df.shape)
powerbi_df.head()

(114, 19)


,Company,cik,form,filed,ddate,Year,Quarter,Revenue,Gross_Profit,Operating_Income,Net_Income,Operating_Cash_Flow,Equity,Profit_Margin,Gross_Margin,Operating_Margin,ROE,Cash_Flow_Margin,Sector
25,Amazon,1018724,10-Q,2021-07-30,2020-03-31,2020,1,NaN,NaN,NaN,NaN,NaN,6.527200e+10,NaN,NaN,NaN,NaN,NaN,Technology
30,Amazon,1018724,10-Q,2021-10-29,2020-06-30,2020,2,NaN,NaN,NaN,NaN,NaN,7.372800e+10,NaN,NaN,NaN,NaN,NaN,Technology
31,Amazon,1018724,10-Q,2021-10-29,2020-09-30,2020,3,2.605090e+11,NaN,6.194000e+09,6.331000e+09,5.529200e+10,8.277500e+10,2.430242,NaN,2.377653,7.648445,21.224603,Technology
45,Amazon,1018724,10-Q,2022-10-28,2020-12-31,2020,4,NaN,NaN,NaN,NaN,NaN,9.340400e+10,NaN,NaN,NaN,NaN,NaN,Technology
40,Amazon,1018724,10-Q,2022-07-29,2021-03-31,2021,1,NaN,NaN,NaN,NaN,NaN,1.033200e+11,NaN,NaN,NaN,NaN,NaN,Technology


In [40]:
powerbi_df.to_csv(
    "PowerBI_SEC_Dashboard_2020_2025_FIXED.csv",
    index=False
)

print("Fixed Power BI file exported successfully")

Fixed Power BI file exported successfully


In [42]:
test = pd.read_csv("PowerBI_SEC_Dashboard_2020_2025_FIXED.csv")

print(test.shape)
print(test["Company"].value_counts())

test[
    test["Company"] == "JPMorgan Chase"
][
    ["Company", "Year", "Quarter", "Revenue", "Net_Income"]
].head(10)

(114, 19)
Company
Amazon            23
JPMorgan Chase    23
Microsoft         23
Tesla             23
Apple             22
Name: count, dtype: int64


,Company,Year,Quarter,Revenue,Net_Income
45,JPMorgan Chase,2020,1,2.828600e+10,2.865000e+09
46,JPMorgan Chase,2020,2,3.307500e+10,7.552000e+09
47,JPMorgan Chase,2020,3,2.925500e+10,1.699500e+10
48,JPMorgan Chase,2020,4,NaN,NaN
49,JPMorgan Chase,2021,1,3.226600e+10,1.430000e+10
50,JPMorgan Chase,2021,2,3.047900e+10,2.624800e+10
51,JPMorgan Chase,2021,3,2.964700e+10,3.793500e+10
52,JPMorgan Chase,2021,4,NaN,NaN
53,JPMorgan Chase,2022,1,3.071700e+10,8.282000e+09
54,JPMorgan Chase,2022,2,3.071500e+10,1.693100e+10
